## Installation Requirements

- A JSL license with **Visual NLP** and **Healthcare NLP** secrets
- **Visual NLP** release **6.4.4 or higher**
- **Healthcare NLP** and **Spark-NLP** releases compatible with your Visual NLP release
- Additional packages: **matplotlib**

### Installation

- Install scripts: https://github.com/JohnSnowLabs/visual-nlp-workshop/tree/master/sh_install_scripts
- Databricks: https://github.com/JohnSnowLabs/visual-nlp-workshop/blob/master/databricks/Readme.md

In [1]:
import json
import os

# Load Credentials from the license file
license = "/content/spark_ocr.json"

if license and "json" in license:

    with open(license, "r") as creds_in:
        creds = json.loads(creds_in.read())

        for key in creds.keys():
            os.environ[key] = creds[key]
else:
    raise Exception("License JSON File is not specified")


# Start Visual NLP Spark session 
from sparkocr import start
import os

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"

extra_configurations = {
    "spark.extraListeners": "com.johnsnowlabs.license.LicenseLifeCycleManager"
}

# jar_path is used Internally for development
spark = start(
    secret = os.environ.get("SPARK_OCR_SECRET"),
    nlp_secret = os.environ.get("SECRET"),
    jar_path = None,
    nlp_internal = os.environ.get("JSL_VERSION"),
    extra_conf=extra_configurations
)

spark

Spark version: 3.4.1
Spark NLP version: 6.4.2
Spark NLP for Healthcare version: 6.4.1
Spark OCR version: 6.4.3rc2

:: loading settings :: url = jar:file:/usr/local/lib/python3.11/dist-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /root/.ivy2/cache
The jars for the packages stored in: /root/.ivy2/jars
com.johnsnowlabs.nlp#spark-nlp-gpu_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-885b4727-e9c9-4f4f-9fd1-8f585abcf8cb;1.0
	confs: [default]
	found com.johnsnowlabs.nlp#spark-nlp-gpu_2.12;6.4.2 in central
	found com.typesafe#config;1.4.2 in central
	found org.rocksdb#rocksdbjni;6.29.5 in central
	found com.amazonaws#aws-java-sdk-s3;1.12.500 in central
	found com.amazonaws#aws-java-sdk-kms;1.12.500 in central
	found com.amazonaws#aws-java-sdk-core;1.12.500 in central
	found commons-logging#commons-logging;1.1.3 in central
	found commons-codec#commons-codec;1.15 in central
	found org.apache.httpcomponents#httpclient;4.5.13 in central
	found org.apache.httpcomponents#httpcore;4.4.13 in central
	found software.amazon.ion#ion-java;1.0.2 in central
	found joda-time#joda-time;2.8.1 in central
	found com.amazonaws#jmespath-java;1.12.500 in central
	fou

## Import Visual-NLP, Healthcare-NLP, Spark-NLP

In [2]:
import os
from textwrap import dedent

from pyspark.sql import functions as F
from pyspark.sql.types import StringType

from sparknlp.annotator import *
from sparknlp.base import *

import sparknlp_jsl
from sparknlp_jsl.annotator import *

import sparkocr
from sparkocr.transformers import *
from sparkocr.utils import *
from sparkocr.enums import *
from sparkocr.schemas import *

## Define VLM OCR 1B 

In [3]:
from sparkocr.transformers.medical_vision_llm import MedicalVisionLLM

ocr_model = MedicalVisionLLM.pretrained("jsl-ocr-gguf-vlm1", "en", "clinical/ocr") \
  .setInputCols(["caption_document", "image_assembler"]) \
  .setOutputCol("completions") \
  .setNGpuLayers(99) \
  .setNCtx(32768) \
  .setNParallel(1) \
  .setNBatch(2048) \
  .setNUbatch(1024) \
  .setNPredict(4096) \
  .setTemperature(0.01) \
  .setTopK(1) \
  .setTopP(1.0) \
  .setRepeatPenalty(1.03) \
  .setRepeatLastN(256) \
  .setStopStrings(["<\uff5chy_Assistant\uff5c>", "<\uff5chy_place\u2581holder\u2581no\u25812\uff5c>"]) \
  .setMinKeep(0) \
  .setNProbs(0) \
  .setOutputCol("completions") \
  .setBatchSize(1) \
  .setDisableLog(False)

jsl-ocr-gguf-vlm1 download started this may take some time.
Approximate size to download 1.5 GB


26/08/24 08:11:35 WARN S3AbortableInputStream: Not all bytes were read from the S3ObjectInputStream, aborting HTTP connection. This is likely an error and may result in sub-optimal behavior. Request only the bytes you need via a ranged GET or drain the input stream after use.
26/08/24 08:11:36 WARN S3AbortableInputStream: Not all bytes were read from the S3ObjectInputStream, aborting HTTP connection. This is likely an error and may result in sub-optimal behavior. Request only the bytes you need via a ranged GET or drain the input stream after use.


jsl-ocr-gguf-vlm1 download started this may take some time.
Approximate size to download 1.5 GB
Download done! Loading the resource.
Extracted 'libjllama.so' to '/tmp/libjllama.so'


ggml_cuda_init: found 1 CUDA devices (Total VRAM: 45498 MiB):
  Device 0: NVIDIA A40, compute capability 8.6, VMM: yes, VRAM: 45498 MiB


## Define Healthcare-NLP De-Identification Text Pipeline

In [4]:
def nlp_deid_pipeline(input_column="text"):
    
    document_assembler = DocumentAssembler() \
        .setInputCol(input_column) \
        .setOutputCol("document") \
        .setCleanupMode("shrink_full")

    sentence_detector = SentenceDetector() \
        .setInputCols(["document"]) \
        .setOutputCol("sentence")
    
    labels = ["NAME", "DATE", "ID", "CONTACT"]
    zeroshot_ner_deid_generic_nonMedical_medium = PretrainedZeroShotNERChunker().pretrained("zeroshot_ner_deid_generic_nonMedical_medium", "en", "clinical/models")\
        .setInputCols("document")\
        .setOutputCol("ner_zeroshot_ner_deid_generic_nonMedical_medium")\
        .setPredictionThreshold(0.5)\
        .setLabels(labels)

    labels = ["IDNUM","MEDICALRECORD", "NAME","PATIENT", "PHONE"]
    zeroshot_ner_deid_subentity_nonMedical_large = PretrainedZeroShotNERChunker().pretrained("zeroshot_ner_deid_subentity_nonMedical_large", "en", "clinical/models")\
        .setInputCols("document")\
        .setOutputCol("ner_zeroshot_ner_deid_subentity_nonMedical_large")\
        .setPredictionThreshold(0.5)\
        .setLabels(labels)

    labels = ['CONTACT', 'DATE', 'ID', 'NAME'] 
    zeroshot_ner_deid_generic_multi_large = PretrainedZeroShotNERChunker().pretrained("zeroshot_ner_deid_generic_multi_large", "xx", "clinical/models")\
        .setInputCols("document")\
        .setOutputCol("ner_zeroshot_ner_deid_generic_multi_large")\
        .setPredictionThreshold(0.5)\
        .setLabels(labels)
    
    date_regex_matcher = RegexMatcherInternalModel.pretrained("date_matcher","en","clinical/models") \
        .setInputCols(["sentence"]) \
        .setOutputCol("date_chunk")
    
    chunk_merger = ChunkMergeApproach()\
        .setInputCols("ner_zeroshot_ner_deid_generic_nonMedical_medium", "ner_zeroshot_ner_deid_subentity_nonMedical_large", "ner_zeroshot_ner_deid_generic_multi_large", "date_chunk")\
        .setOutputCol('merged_chunk')\
        .setMergeOverlapping(True)

    nlp_pipeline = Pipeline(stages=[
        document_assembler,
        sentence_detector,
        zeroshot_ner_deid_generic_nonMedical_medium,
        zeroshot_ner_deid_subentity_nonMedical_large,
        zeroshot_ner_deid_generic_multi_large,
        date_regex_matcher,
        chunk_merger
    ])

    empty_data = spark.createDataFrame([[""]]).toDF(input_column)
    nlp_model = nlp_pipeline.fit(empty_data)
    return nlp_model

## Wrap DICOM Ingestion / Finalizer stages around the de-identification pipeline

The de-identification pipeline ends with **`ImageDrawRegions`**, which redacts the burned-in pixel data. To emit fully de-identified DICOM files, we should add below stages at the end:

- **`DicomDrawRegions`** — applies the redaction regions back onto the DICOM pixel data
- **`DicomMetadataDeIdentifier`** — de-identifies the DICOM metadata / header tags

In [5]:
scale = 1.0

dicom_to_image = DicomToImageV3() \
    .setInputCols(["content"]) \
    .setOutputCol("image") \
    .setCompressImage(True) \
    .setCompressionMode("enabled") \
    .setKeepInput(False) \
    .setMemoryOptimized(True) \
    .setCompressionQuality(80) \
    .setScale(scale)

caption_assembler = DocumentAssembler() \
    .setInputCol("caption") \
    .setOutputCol("caption_document")

schema_converter_assembler = ImageSchemaConverter() \
    .setInputCol("image") \
    .setOutputCol("image_assembler") \
    .setOutputSchema("assembler") \
    .setKeepInput(False)

coordinate_extract = DocumentCoordinatesToText() \
    .setInputCol("completions") \
    .setImageDimsCol("frame_dims") \
    .setOutputCol("text") \
    .setPageMatrixCol("positions") \
    .setRegionCol("regions")

position_finder = PositionFinder() \
    .setInputCols(["merged_chunk"]) \
    .setOutputCol("coordinates") \
    .setPageMatrixCol("positions") \
    .setIgnoreSchema(True) \
    .setOcrScaleFactor(1.0)

schema_converter_internal = ImageSchemaConverter() \
    .setInputCol("image_assembler") \
    .setOutputCol("image") \
    .setOutputSchema("internal") \
    .setKeepInput(False)

draw_regions = ImageDrawRegions() \
    .setInputCol("image") \
    .setInputRegionsCol("coordinates") \
    .setRectColor(Color.black) \
    .setFilledRect(True) \
    .setOutputCol("image_with_regions")

pipeline = PipelineModel(stages=[
    dicom_to_image,
    caption_assembler,
    schema_converter_assembler,
    ocr_model,
    coordinate_extract,
    nlp_deid_pipeline(input_column="text"),
    position_finder,
    schema_converter_internal,
    draw_regions
])

zeroshot_ner_deid_generic_nonMedical_medium download started this may take some time.
Approximate size to download 753.2 MB
[ | ]

26/08/24 08:12:17 WARN S3AbortableInputStream: Not all bytes were read from the S3ObjectInputStream, aborting HTTP connection. This is likely an error and may result in sub-optimal behavior. Request only the bytes you need via a ranged GET or drain the input stream after use.
26/08/24 08:12:18 WARN S3AbortableInputStream: Not all bytes were read from the S3ObjectInputStream, aborting HTTP connection. This is likely an error and may result in sub-optimal behavior. Request only the bytes you need via a ranged GET or drain the input stream after use.


zeroshot_ner_deid_generic_nonMedical_medium download started this may take some time.
Approximate size to download 753.2 MB
[ / ]Download done! Loading the resource.


[ — ]

[OK!]
zeroshot_ner_deid_subentity_nonMedical_large download started this may take some time.


...

Approximate size to download 2.4 KB
[ | ]

26/08/24 08:13:21 WARN S3AbortableInputStream: Not all bytes were read from the S3ObjectInputStream, aborting HTTP connection. This is likely an error and may result in sub-optimal behavior. Request only the bytes you need via a ranged GET or drain the input stream after use.
26/08/24 08:13:22 WARN S3AbortableInputStream: Not all bytes were read from the S3ObjectInputStream, aborting HTTP connection. This is likely an error and may result in sub-optimal behavior. Request only the bytes you need via a ranged GET or drain the input stream after use.


date_matcher download started this may take some time.
Approximate size to download 2.4 KB
Download done! Loading the resource.


[ / ]

[OK!]


## Load Synthetic Dicom Files

In [7]:
vision_prompt = "Detect and recognize text in the image, and output the text coordinates in a formatted manner."
path = "./data/original/dicom/pixel/*"

df = spark.read.format("binaryFile").load(path) \
    .withColumn("caption", F.lit(vision_prompt)) \
    .withColumn("path", F.regexp_replace(F.col("path"), "dbfs:", ""))

print(f"Total DICOM Files : {df.count()}")

Total DICOM Files : 40


## Generate Results

In [8]:
result = pipeline.transform(df).cache()
result.columns

/usr/local/lib/python3.11/dist-packages/pyspark/sql/dataframe.py:169: UserWarning: DataFrame.sql_ctx is an internal property, and will be removed in future releases. Use DataFrame.sparkSession instead.
  warnings.warn(
26/08/24 08:13:56 WARN package: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


['pagenum',
 'frame_dims',
 'path',
 'modificationTime',
 'length',
 'caption',
 'caption_document',
 'completions',
 'text',
 'regions',
 'positions',
 'document',
 'sentence',
 'ner_zeroshot_ner_deid_generic_nonMedical_medium',
 'ner_zeroshot_ner_deid_subentity_nonMedical_large',
 'ner_zeroshot_ner_deid_generic_multi_large',
 'date_chunk',
 'merged_chunk',
 'coordinates',
 'image',
 'image_with_regions',
 'exception']

## Save Deid Images To Disk

In [9]:
root_path = "./data/deid/image/pixel/"

os.makedirs(root_path, exist_ok=True)

for item in result.select("path", "image_with_regions").toLocalIterator():
    
    filename = os.path.basename(item.path).replace(".dcm", ".png")
    
    img = to_pil_image(item.image_with_regions, item.image_with_regions.mode)
    img_save_path = os.path.join(root_path, filename)
    
    img.save(img_save_path)

26/08/24 08:15:30 WARN DAGScheduler: Broadcasting large task binary with size 2.2 MiB
08:15:38, INFO Run DicomToImageV3                                   (0 + 1) / 1]
08:15:45, INFO DicomToImageV3: Number of frames: 1
08:15:45, INFO DicomToImageV3: Number of frames To Extract: 0
08:15:45, INFO DicomToImageV3: Extracting All Frames.
08:15:46, INFO Run DicomToImageV3
08:15:46, INFO DicomToImageV3: Number of frames: 1
08:15:46, INFO DicomToImageV3: Number of frames To Extract: 0
08:15:46, INFO DicomToImageV3: Extracting All Frames.
08:15:46, INFO Run DicomToImageV3
08:15:46, INFO DicomToImageV3: Number of frames: 1
08:15:46, INFO DicomToImageV3: Number of frames To Extract: 0
08:15:46, INFO DicomToImageV3: Extracting All Frames.
08:15:46, INFO Run DicomToImageV3
08:15:46, INFO DicomToImageV3: Number of frames: 1
08:15:46, INFO DicomToImageV3: Number of frames To Extract: 0
08:15:46, INFO DicomToImageV3: Extracting All Frames.
08:15:46, INFO Run DicomToImageV3
08:15:46, INFO DicomToImageV3

## Save de-identified DICOM [ pixels + metadata ] to disk

We already have the pixel coordinates that were passed to `ImageDrawRegions` in the previous pipeline. We reuse those same coordinates in `DicomDrawRegions`, which renders a DICOM file with the pixels redacted. That new file is then passed to `DicomMetadataDeidentifier`, which de-identifies the metadata.

The result holds the fully de-identified DICOM bytes in `dicom_metadata_cleaned` — pixels redacted and metadata cleaned — ready to write to disk.

In [11]:
draw_regions = DicomDrawRegions() \
    .setInputCol("path") \
    .setInputRegionsCol("coordinates") \
    .setOutputCol("dicom") \
    .setAggCols(["path"]) \
    .setKeepInput(True) \
    .setScaleFactor(1 / scale)

strategy_file_path = "./dicom_metadata_deidentification_strategy.csv"

dicom_deidentifier = DicomMetadataDeidentifier() \
    .setInputCols(["dicom"]) \
    .setOutputCol("dicom_meta_cleaned") \
    .setKeepInput(False) \
    .setRemovePrivateTags(False) \
    .setStrategyFile(strategy_file_path)

pipeline = PipelineModel(stages=[
    draw_regions,
    dicom_deidentifier
])

## Save Deid DICOM To Disk

In [12]:
root_path = "./data/deid/dicom/pixel/"

os.makedirs(root_path, exist_ok=True)

dicom_result = pipeline.transform(result)

for item in dicom_result.select("path", "dicom_meta_cleaned").toLocalIterator():
    data = item.asDict()
    filename = os.path.basename(data["path"])

    file_out_path = os.path.join(root_path, filename)

    with open(file_out_path, "wb") as dicom_out:
        dicom_out.write(data["dicom_meta_cleaned"])

/usr/local/lib/python3.11/dist-packages/pyspark/sql/dataframe.py:169: UserWarning: DataFrame.sql_ctx is an internal property, and will be removed in future releases. Use DataFrame.sparkSession instead.
  warnings.warn(
26/08/24 08:18:06 WARN DAGScheduler: Broadcasting large task binary with size 2.1 MiB
26/08/24 08:18:07 WARN DAGScheduler: Broadcasting large task binary with size 2.1 MiB
26/08/24 08:18:08 WARN DAGScheduler: Broadcasting large task binary with size 2.2 MiB
08:18:08, INFO DicomDrawRegions: file:/workspace/Synthetic/Synthetic_Dataset/data/original/dicom/pixel/Synthetic_Pixel_015.dcm
08:18:08, INFO DicomDrawRegions: Original Size: 1049886             (0 + 1) / 1]
08:18:08, WARNING DicomDrawRegions: Dicom does not contain NumberOfFrames Tag Assuming value to be 1.
08:18:08, INFO DicomDrawRegions: Photometric Interpretation: MONOCHROME2
08:18:08, INFO DicomDrawRegions: Transfer Syntax: 1.2.840.10008.1.2.1
08:18:08, INFO DicomDrawRegions: Pixel Representation: 0
08:18:08, INF

## Define OCR and metadata extraction pipeline

Now that we have the de-identified DICOM, we run OCR again to extract the text after de-identification. This confirms which burned-in pixels were removed — the previously detected PHI should no longer show up.

We also use `DicomToMetadata` to extract the tags from the de-identified DICOM file, so we can verify the metadata was cleaned.

In [13]:
dicom_to_image = DicomToImageV3() \
    .setInputCols(["content"]) \
    .setOutputCol("image") \
    .setCompressImage(True) \
    .setCompressionMode("enabled") \
    .setKeepInput(False) \
    .setMemoryOptimized(True) \
    .setCompressionQuality(80) \
    .setScale(scale)

caption_assembler = DocumentAssembler() \
    .setInputCol("caption") \
    .setOutputCol("caption_document")

schema_converter_assembler = ImageSchemaConverter() \
    .setInputCol("image") \
    .setOutputCol("image_assembler") \
    .setOutputSchema("assembler") \
    .setKeepInput(False)

coordinate_extract = DocumentCoordinatesToText() \
    .setInputCol("completions") \
    .setImageDimsCol("frame_dims") \
    .setOutputCol("text") \
    .setPageMatrixCol("positions") \
    .setRegionCol("regions")

dicom_to_metadata = DicomToMetadata() \
    .setInputCol("path") \
    .setOutputCol("metadata") \
    .setKeepInput(True) \
    .setExtractTagForNer(False)

pipeline = PipelineModel(stages=[
    dicom_to_image,
    caption_assembler,
    schema_converter_assembler,
    ocr_model,
    coordinate_extract,
    dicom_to_metadata
])

### Load the Final Deid Dicom

In [14]:
vision_prompt = "Detect and recognize text in the image, and output the text coordinates in a formatted manner."
path = "./data/deid/dicom/pixel/*"

df = spark.read.format("binaryFile").load(path) \
    .withColumn("caption", F.lit(vision_prompt)) \
    .withColumn("path", F.regexp_replace(F.col("path"), "dbfs:", ""))

print(f"Total De-Identified DICOM Files : {df.count()}")

Total De-Identified DICOM Files : 40


### Generate Result

In [15]:
result = pipeline.transform(df).cache()

deid_result = result.select("path", "text", "metadata").collect()

/usr/local/lib/python3.11/dist-packages/pyspark/sql/dataframe.py:169: UserWarning: DataFrame.sql_ctx is an internal property, and will be removed in future releases. Use DataFrame.sparkSession instead.
  warnings.warn(
08:20:00, INFO Run DicomToImageV3                                   (0 + 1) / 2]
08:20:06, INFO DicomToImageV3: Number of frames: 1
08:20:06, INFO DicomToImageV3: Number of frames To Extract: 0
08:20:06, INFO DicomToImageV3: Extracting All Frames.
08:20:07, INFO Run DicomToImageV3
08:20:07, INFO DicomToImageV3: Number of frames: 1
08:20:07, INFO DicomToImageV3: Number of frames To Extract: 0
08:20:07, INFO DicomToImageV3: Extracting All Frames.
08:20:07, INFO Run DicomToImageV3
08:20:07, INFO DicomToImageV3: Number of frames: 1
08:20:07, INFO DicomToImageV3: Number of frames To Extract: 0
08:20:07, INFO DicomToImageV3: Extracting All Frames.
08:20:07, INFO Run DicomToImageV3
08:20:07, INFO DicomToImageV3: Number of frames: 1
08:20:07, INFO DicomToImageV3: Number of frame

### Collect OCR + Metadata Result

In [87]:
deid_result_mapping = {}

for item in deid_result:
    basename = os.path.basename(item.path)
    ocr_text = item.text
    metadata = json.loads(item.metadata)

    deid_result_mapping[basename] = {"ocr" : ocr_text, "metadata": metadata}

print(f"Total De-Identified DICOM Files Result: {len(deid_result_mapping.keys())}")

Total De-Identified DICOM Files Result: 40


In [88]:
list(deid_result_mapping.keys())[:5]

['Synthetic_Pixel_017.dcm',
 'Synthetic_Pixel_001.dcm',
 'Synthetic_Pixel_015.dcm',
 'Synthetic_Pixel_021.dcm',
 'Synthetic_Pixel_025.dcm']

In [89]:
deid_result_mapping["Synthetic_Pixel_017.dcm"]

{'ocr': 'CT AXIAL\t120 kV\nSeries: 3\t250 mA\nImage: 87\tFOV 360 mm\nSlice: 5.0 mm\tZoom 1.00\nW 400 L 40\nR\tL',
 'metadata': {'00080016': {'tag': '00080016',
   'vr': 'UI',
   'value': '1.2.840.10008.5.1.4.1.1.1'},
  '00080018': {'tag': '00080018',
   'vr': 'UI',
   'value': '2.25.332727515484471787618534824755661741786'},
  '00080020': {'tag': '00080020', 'vr': 'DA', 'value': '20260609'},
  '00080030': {'tag': '00080030', 'vr': 'TM', 'value': '043959'},
  '00080050': {'tag': '00080050',
   'vr': 'SH',
   'value': '2.25.306688294317502938854716986014619274545'},
  '00080060': {'tag': '00080060', 'vr': 'CS', 'value': 'MR'},
  '00080080': {'tag': '00080080', 'vr': 'LO', 'value': 'anonymous'},
  '00080090': {'tag': '00080090',
   'vr': 'PN',
   'value': 'ortega-fernandez kathleen'},
  '00081030': {'tag': '00081030', 'vr': 'LO', 'value': 'anonymous'},
  '0008103E': {'tag': '0008103E', 'vr': 'LO', 'value': 'anonymous'},
  '00100010': {'tag': '00100010', 'vr': 'PN', 'value': 'aukusti blots

### Compare against the Ground truth

In [90]:
with open("./Image_Ground_Truth.json", "r") as file:
    payload = json.load(file)["dicom_files"]

In [91]:
def correct_tag(tag):
    return tag.replace("(", "").replace(")", "").replace(",", "").replace(" ", "").strip()

def correct_phi(text):
    return str(text).replace("^", "").replace(" ", "").lower().strip()

In [113]:
ground_truth = {}

for dicom_file_gt in payload:
    path = dicom_file_gt["file"]
    
    ground_truth[path] = {"ocr": [], "metadata": {} }

    # Extract Pixel Ground Truth
    for pixel_phi_item in dicom_file_gt["pixel"]["phi"]:
        ground_truth[path]["ocr"].append(correct_phi(pixel_phi_item))

    # Extract Metadata Ground Truth
    for metadata_item in dicom_file_gt["metadata"]:
        
        parent_tag = correct_tag(metadata_item["tag"])
        
        # Check whether the tag is marked to contain phi in gt
        if metadata_item["contains_phi"]:

            # Non SQ Element
            if metadata_item["vr"] != "SQ":
                
                vr = metadata_item["vr"]
                print(type(
                values = [correct_phi(item) for item in metadata_item["phi"]]

                ground_truth[path]["metadata"][parent_tag] = {"tag": parent_tag, "vr": vr, "value": values}
                
            # SQ Element
            else:

                for nested_metadata_item in metadata_item["value"][0]["metadata"]:
                    
                    if nested_metadata_item["contains_phi"]:
                        
                        nested_tag = correct_tag(nested_metadata_item["tag"])
                        complete_tag = f"{parent_tag}[0].{nested_tag}"

                        vr = nested_metadata_item["vr"]
                        values = [correct_phi(item) for item in nested_metadata_item["phi"]]

                        ground_truth[path]["metadata"][complete_tag] = {"tag": nested_tag, "vr": vr, "value": values}

print(f"Total Ground Truth Files: {len(ground_truth.keys())}")

Total Ground Truth Files: 40


In [147]:
ground_truth["Synthetic_Pixel_017.dcm"]

{'ocr': ['rosenelliot', '4953407', '20260702'],
 'metadata': {'00100010': {'tag': '00100010',
   'vr': 'PN',
   'value': ['rosenelliot']},
  '00100020': {'tag': '00100020', 'vr': 'LO', 'value': ['4953407']},
  '00100030': {'tag': '00100030', 'vr': 'DA', 'value': ['19700217']},
  '00101040': {'tag': '00101040',
   'vr': 'LO',
   'value': ['1008willowbendave,nashville,tn37205']},
  '00102154': {'tag': '00102154', 'vr': 'SH', 'value': ['555-015-4936']},
  '00080050': {'tag': '00080050', 'vr': 'SH', 'value': ['acc-20260702-017']},
  '00080020': {'tag': '00080020', 'vr': 'DA', 'value': ['20260702']},
  '00080030': {'tag': '00080030', 'vr': 'TM', 'value': ['150822']},
  '00080080': {'tag': '00080080',
   'vr': 'LO',
   'value': ['willowbendimagingcenter']},
  '00080090': {'tag': '00080090', 'vr': 'PN', 'value': ['davisnoah']},
  '0020000D': {'tag': '0020000D',
   'vr': 'UI',
   'value': ['1.2.826.0.1.3680043.10.54321.20260702.17']},
  '0020000E': {'tag': '0020000E',
   'vr': 'UI',
   'value'

## Final Result

In [149]:
metadata_passed = 0
metadata_failed = 0

pixel_phi_passed = 0
pixel_phi_failed = 0

for key, gt in ground_truth.items():

    # Check Pixel PHI
    gt_ocr = gt["ocr"]
    deid_ocr = correct_phi(deid_result_mapping[key]["ocr"])

    for item in gt_ocr:
        if item in deid_ocr:
            pixel_phi_failed += 1
        else:
            pixel_phi_passed += 1


    # Check Metadata PHI
    gt_metadata = gt["metadata"]
    deid_metadata = deid_result_mapping[key]["metadata"]

    # {'00100010': {'tag': '00100010', 'vr': 'PN', 'value': ['walkernoah', '22081998']}
    for tag, tag_item in gt_metadata.items():
        # ['walkernoah', '22081998']
        for phi in tag_item["value"]:
            
            if phi in str(deid_metadata[tag]["value"]):
                metadata_failed += 1
            else:
                metadata_passed += 1

print(f"Final De-Identification Report {len(ground_truth.keys())} DICOM Files:\n")
print(f"Metadata Passed:  {metadata_passed}")
print(f"Metadata Failed: {metadata_failed}\n")
print(f"Pixel PHI Passed: {pixel_phi_passed}")
print(f"Pixel PHI Failed: {pixel_phi_failed}")

Final De-Identification Report 40 DICOM Files:

Metadata Passed:  1040
Metadata Failed: 0

Pixel PHI Passed: 120
Pixel PHI Failed: 0
